# Fine-tune BARTpho: BTC train → dev mới

Train bằng JSON BTC đã preprocessing; đánh giá trên **toàn bộ file dev mới có nhãn**. Model sinh cả `error_type` và `corrected_sentence` như notebook gốc.

1. Bật **GPU** và **Internet**, thêm hai dataset BTC/dev vào Kaggle.
2. Ở cell cấu hình, gán `BTC_PATH` và `DEV_PATH` nếu tên file không tự nhận diện được. Tên mặc định BTC: `preprocess_btc.json` hoặc `preprocessed_btc.json`; dev: `dev.json`, `vgec_dev.json`, `vgec2026_dev.json`, `vgec_2026_dev.json`. Nếu tên khác, dùng đường dẫn cụ thể.
3. Chạy lần lượt hoặc **Run All**. Train/dev là JSON array theo schema `ID`, `Input`, `Output.Error_Type_Id`, `Output.Fixed Sentence`.
4. Cell dữ liệu xuất số mẫu bị loại khỏi train và danh sách ID. Cell cuối lưu model, prediction, metrics và ZIP trong `/kaggle/working/bartpho_btc_dev/`.

**Quy tắc:** loại khỏi bản train mọi `Input` có trong dev khi so khớp NFC, khoảng trắng và hoa/thường; giữ dấu câu khi so khớp. Không sửa nhãn, câu nguồn, đáp án, dấu `...` hoặc file nguồn. Không tự loại câu gần giống/paraphrase. Toàn bộ đáp án của mỗi mẫu giữ trong một tập; train chọn một đáp án có seed ở mỗi epoch, dev giữ mọi đáp án. Nhãn 0 dùng Input làm target nội bộ, không ghi lại gold `[]` thành câu nguồn.

Có lỗi cấu trúc JSON hoặc nhãn có lỗi nhưng thiếu câu sửa thực sự: dừng và báo để xử lý thủ công. Các cảnh báo tự động được lưu ở `data_issues.json`, không tự sửa gold. Các chỉ số correction là implementation tham khảo của notebook gốc, **không phải scorer chính thức BTC**. Không chia folds, không tính OOF, không chạy baseline so sánh.

Checkpoint tốt nhất được chọn theo dev loss (đáp án đầu tiên). Dev tham gia chọn model nên điểm dev là điểm validation; muốn điểm test độc lập cần tập khác chưa dùng để chọn model.


## 1. Cài thư viện

In [1]:
%pip install -q "transformers==4.57.1" "accelerate==1.10.1" sentencepiece protobuf safetensors "scikit-learn>=1.3,<2"
print("Đã cài thư viện. Nếu trước đó đã import Transformers bản khác, Restart Session rồi Run All.", flush=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 62.6 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Đã cài thư viện. Nếu trước đó đã import Transformers bản khác, Restart Session rồi Run All.


## 2. Cấu hình đường dẫn và training

In [2]:
from pathlib import Path
NOTEBOOK_CODE_SHA256 = 'b5e7015c523218da071908131d3e509bb3b2fbf9d7d224cb491af0cea7747c60'
# CHỈNH HAI ĐƯỜNG DẪN NÀY nếu tên file khác các tên tự nhận diện bên dưới.
# Ví dụ: Path('/kaggle/input/datasets/<user>/<dataset>/preprocessed_btc.json')
BTC_PATH = Path('/kaggle/input/datasets/tuandatdao/combine/train_combine.json')
DEV_PATH = Path('/kaggle/input/datasets/tuandatdao/combine/dev.json')

MODEL_NAME = 'vinai/bartpho-syllable-base'
SEED = 42
EPOCHS = 15
LEARNING_RATE = 3e-5
TRAIN_BATCH_SIZE = 2
GRAD_ACCUMULATION = 8
EVAL_BATCH_SIZE = 2
EARLY_STOPPING_PATIENCE = 3
NUM_BEAMS = 4
FP16 = False
BF16 = False
RESUME_IF_AVAILABLE = True
INCLUDE_CHECKPOINTS_IN_ZIP = False

WORK_ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / 'bartpho_btc_dev'
PIPELINE_VERSION = 'btc-dev-joint-2026-10-09-v1'
print('Đã nạp cấu hình. Cell dữ liệu tiếp theo sẽ kiểm tra và lọc train.', flush=True)


Đã nạp cấu hình. Cell dữ liệu tiếp theo sẽ kiểm tra và lọc train.


## 3. Kiểm tra JSON và loại Input trong dev khỏi train

In [3]:
import copy
import hashlib
import json
import re
import shutil
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'))

def sha256_bytes(payload):
    return hashlib.sha256(payload).hexdigest()

def sentence_key(text):
    # Chỉ dùng để SO KHỚP: không sửa chuỗi lưu trong dữ liệu.
    return ' '.join(unicodedata.normalize('NFC', text).split()).casefold()

def resolve_input_path(configured, role):
    if configured is not None:
        result = Path(configured).expanduser().resolve()
        if not result.is_file():
            raise FileNotFoundError(f'{role}: không thấy {result}. Sửa đường dẫn ở cell cấu hình.')
        return result
    names = ({'preprocess_btc.json', 'preprocessed_btc.json'} if role == 'BTC'
             else {'dev.json', 'vgec_dev.json', 'vgec2026_dev.json', 'vgec_2026_dev.json'})
    candidates = set()
    roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [Path.cwd() / 'upload', Path.cwd()]
    for root in roots:
        if not root.is_dir():
            continue
        paths = root.rglob('*.json') if str(root) == '/kaggle/input' else root.glob('*.json')
        for path in paths:
            if path.name.casefold() in names:
                candidates.add(path.resolve())
    if len(candidates) != 1:
        found = '\n'.join(str(p) for p in sorted(candidates)) or '(không tìm thấy)'
        raise ValueError(f'{role}: cần đúng một file; tìm thấy {len(candidates)}.\n{found}\n'
                         f'Gán {role}_PATH = Path("/kaggle/input/.../ten_file.json") ở cell cấu hình.')
    return candidates.pop()

def load_gold_json(path, role):
    payload = Path(path).read_bytes()
    def object_pairs(pairs):
        result = {}
        for key, value in pairs:
            if key in result:
                raise ValueError(f'{role}: khóa JSON lặp: {key!r}')
            result[key] = value
        return result
    def reject_constant(value):
        raise ValueError(f'{role}: giá trị JSON không hợp lệ: {value}')
    records = json.loads(payload.decode('utf-8-sig'), object_pairs_hook=object_pairs, parse_constant=reject_constant)
    if not isinstance(records, list) or not records:
        raise ValueError(f'{role}: cần một JSON array không rỗng theo schema BTC.')
    seen_ids = set()
    for position, row in enumerate(records, 1):
        prefix = f'{role}, vị trí {position}'
        if not isinstance(row, dict) or not {'ID', 'Input', 'Output'} <= set(row):
            raise ValueError(f'{prefix}: thiếu ID/Input/Output.')
        if type(row['ID']) is not int or row['ID'] in seen_ids:
            raise ValueError(f'{prefix}: ID phải là số nguyên duy nhất trong từng file.')
        seen_ids.add(row['ID'])
        if not isinstance(row['Input'], str) or not sentence_key(row['Input']):
            raise ValueError(f'{prefix}: Input phải là chuỗi không rỗng.')
        output = row['Output']
        if not isinstance(output, dict) or not {'Error_Type_Id', 'Fixed Sentence'} <= set(output):
            raise ValueError(f'{prefix}: Output sai schema.')
        if type(output['Error_Type_Id']) is not int or output['Error_Type_Id'] not in range(5):
            raise ValueError(f'{prefix}: nhãn phải là số nguyên 0..4.')
        if not isinstance(output['Fixed Sentence'], list) or any(
                not isinstance(ref, str) or not sentence_key(ref) for ref in output['Fixed Sentence']):
            raise ValueError(f'{prefix}: Fixed Sentence phải là list chuỗi không rỗng (list [] được phép).')
    return records, payload

def group_duplicates(records):
    groups = defaultdict(list)
    for row in records:
        groups[sentence_key(row['Input'])].append(row['ID'])
    return [ids for ids in groups.values() if len(ids) > 1]

def filter_train_against_dev(btc, dev):
    dev_by_key = defaultdict(list)
    for row in dev:
        dev_by_key[sentence_key(row['Input'])].append(row['ID'])
    kept, removed, log = [], [], []
    for position, row in enumerate(btc, 1):
        key = sentence_key(row['Input'])
        if key in dev_by_key:
            removed.append(copy.deepcopy(row))
            log.append({'btc_ID': row['ID'], 'btc_position': position, 'dev_IDs': dev_by_key[key],
                        'reason': 'Input trùng sau NFC + gộp khoảng trắng + casefold'})
        else:
            kept.append(copy.deepcopy(row))
    if not kept:
        raise ValueError('Sau khi loại các Input có trong dev, tập train rỗng. Kiểm tra lại hai file.')
    if not {sentence_key(r['Input']) for r in kept}.isdisjoint(dev_by_key):
        raise RuntimeError('Kiểm tra nội bộ thất bại: train/dev còn Input trùng.')
    return kept, removed, log

def reference_issues(records, role):
    issues = []
    for row in records:
        label, refs = row['Output']['Error_Type_Id'], row['Output']['Fixed Sentence']
        key = sentence_key(row['Input'])
        if label == 0 and any(sentence_key(ref) != key for ref in refs):
            issues.append({'severity': 'fatal', 'reason': 'Nhãn 0 nhưng reference khác Input', 'dataset': role, 'record': row})
        if label != 0 and (not refs or all(sentence_key(ref) == key for ref in refs)):
            issues.append({'severity': 'fatal', 'reason': 'Có lỗi nhưng thiếu reference sửa thực sự', 'dataset': role, 'record': row})
        elif label != 0 and any(sentence_key(ref) == key for ref in refs):
            issues.append({'severity': 'warning', 'reason': 'Một reference giống Input; giữ nguyên để người dùng rà soát', 'dataset': role, 'record': row})
        for index, ref in enumerate(refs):
            if ref.count('“') != ref.count('”'):
                issues.append({'severity': 'warning', 'reason': 'Ngoặc kép cong không cân bằng', 'reference_index': index, 'dataset': role, 'record': row})
    return issues

def to_model_records(records, dataset_name):
    result = []
    for row in records:
        label = row['Output']['Error_Type_Id']
        refs = [row['Input']] if label == 0 else list(row['Output']['Fixed Sentence'])
        result.append({'id': row['ID'], 'dataset': dataset_name, 'source': row['Input'],
                       'error_type': label, 'references': refs})
    return result

def prepare_btc_dev(btc_path, dev_path, output_root):
    if Path(btc_path).resolve() == Path(dev_path).resolve():
        raise ValueError('BTC_PATH và DEV_PATH đang chỉ tới cùng một file.')
    btc, btc_bytes = load_gold_json(btc_path, 'BTC')
    dev, dev_bytes = load_gold_json(dev_path, 'DEV')
    train, removed, removal_log = filter_train_against_dev(btc, dev)
    hashes = {'btc': sha256_bytes(btc_bytes), 'dev': sha256_bytes(dev_bytes)}
    pair_hash = sha256_bytes(json.dumps(hashes, sort_keys=True).encode())
    data_dir = Path(output_root) / 'data' / pair_hash
    data_dir.mkdir(parents=True, exist_ok=True)
    (data_dir / 'btc_original.json').write_bytes(btc_bytes)
    (data_dir / 'dev_original.json').write_bytes(dev_bytes)
    issues = reference_issues(train, 'BTC_train') + reference_issues(dev, 'DEV')
    write_json(data_dir / 'train_filtered.json', train)
    write_json(data_dir / 'dev_preserved.json', dev)
    write_json(data_dir / 'removed_train_dev_overlap.json', removed)
    write_json(data_dir / 'overlap_log.json', removal_log)
    write_json(data_dir / 'data_issues.json', issues)
    manifest = {'input_paths': {'btc': str(btc_path), 'dev': str(dev_path)}, 'sha256': hashes,
                'btc_before': len(btc), 'train_after': len(train), 'removed_overlap': len(removed),
                'dev_count': len(dev), 'match_rule': 'NFC + whitespace collapse + casefold on Input only; punctuation preserved',
                'train_labels': dict(sorted(Counter(r['Output']['Error_Type_Id'] for r in train).items())),
                'dev_labels': dict(sorted(Counter(r['Output']['Error_Type_Id'] for r in dev).items())),
                'duplicates_train_IDs': group_duplicates(train), 'duplicates_dev_IDs': group_duplicates(dev),
                'train_dev_remaining_Input_overlap': 0, 'gold_content_modified': False,
                'warning_count': sum(i['severity'] == 'warning' for i in issues),
                'fatal_count': sum(i['severity'] == 'fatal' for i in issues),
                'near_duplicates': 'Không tự loại paraphrase/câu gần giống hoặc khác dấu câu; cần rà soát riêng.'}
    write_json(data_dir / 'data_manifest.json', manifest)
    # Bảo đảm file nguồn và toàn bộ nội dung các hàng còn lại không bị sửa.
    if Path(btc_path).read_bytes() != btc_bytes or Path(dev_path).read_bytes() != dev_bytes:
        raise RuntimeError('File nguồn thay đổi trong khi chuẩn bị; chạy lại với file ổn định.')
    assert len(train) + len(removed) == len(btc)
    assert train == [r for r in btc if sentence_key(r['Input']) not in {sentence_key(d['Input']) for d in dev}]
    assert read_json(data_dir / 'dev_preserved.json') == dev
    if manifest['fatal_count']:
        raise ValueError(f"Có {manifest['fatal_count']} mẫu không dùng được để train/eval; xem {data_dir / 'data_issues.json'} rồi xử lý thủ công. Code không sửa hoặc loại thêm gold.")
    return {'data_dir': data_dir, 'manifest': manifest, 'train_raw': train, 'dev_raw': dev,
            'train_records': to_model_records(train, 'BTC_train'), 'dev_records': to_model_records(dev, 'DEV')}

print('Đang tìm hai file JSON...', flush=True)
BTC_PATH = resolve_input_path(BTC_PATH, 'BTC')
DEV_PATH = resolve_input_path(DEV_PATH, 'DEV')
prepared = prepare_btc_dev(BTC_PATH, DEV_PATH, OUTPUT_ROOT)
DATA_DIR = prepared['data_dir']
train_records, dev_records = prepared['train_records'], prepared['dev_records']
print(json.dumps(prepared['manifest'], ensure_ascii=False, indent=2), flush=True)
print('Đã lưu train sau lọc, dev nguyên nội dung, các mẫu loại và báo cáo tại:', DATA_DIR, flush=True)


Đang tìm hai file JSON...
{
  "input_paths": {
    "btc": "/kaggle/input/datasets/tuandatdao/combine/train_combine.json",
    "dev": "/kaggle/input/datasets/tuandatdao/combine/dev.json"
  },
  "sha256": {
    "btc": "56041858cc83626971a0e9c81abae40cf8ea6e21b696a5e040e4b98e1ea29787",
    "dev": "9ecdd1529e28264f6e6c789cf49da48ac5caf4ed92509d09201fc77a17fb7037"
  },
  "btc_before": 1857,
  "train_after": 1857,
  "removed_overlap": 0,
  "dev_count": 90,
  "match_rule": "NFC + whitespace collapse + casefold on Input only; punctuation preserved",
  "train_labels": {
    "0": 486,
    "1": 418,
    "2": 319,
    "3": 308,
    "4": 326
  },
  "dev_labels": {
    "0": 42,
    "1": 29,
    "2": 6,
    "3": 6,
    "4": 7
  },
  "duplicates_train_IDs": [],
  "duplicates_dev_IDs": [],
  "train_dev_remaining_Input_overlap": 0,
  "gold_content_modified": false,
  "warning_count": 1,
  "fatal_count": 0,
  "near_duplicates": "Không tự loại paraphrase/câu gần giống hoặc khác dấu câu; cần rà soát riêng.

## 4. Tokenizer, dataset, định dạng output và parser

In [4]:
import gc
import math
import os
import random

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
import torch
import transformers
from packaging.version import Version
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback,
                          EarlyStoppingCallback, set_seed)

if transformers.__version__ != '4.57.1':
    raise RuntimeError('Cần transformers==4.57.1. Chạy cell cài đặt rồi Restart Session nếu đã import bản khác.')
if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU trong Kaggle Settings trước khi chạy các cell model/train.')
if Version(torch.__version__.split('+')[0]) < Version('2.6'):
    raise RuntimeError('Cần PyTorch >= 2.6 để đọc checkpoint .bin gốc. Dùng image Kaggle GPU có phiên bản phù hợp.')
if FP16 and BF16:
    raise ValueError('Không bật đồng thời FP16 và BF16.')
if BF16 and not torch.cuda.is_bf16_supported():
    raise ValueError('GPU hiện tại không hỗ trợ BF16; đặt BF16=False.')
set_seed(SEED)
print('GPU:', torch.cuda.get_device_name(0), '| GPU count:', torch.cuda.device_count(), flush=True)
print('Torch:', torch.__version__, '| Transformers:', transformers.__version__, flush=True)
print('Đang tải tokenizer...', flush=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def make_joint_target(label, correction):
    return f'error_type: {label} corrected_sentence: {correction}'

class CorrectionDataset(Dataset):
    def __init__(self, records, tokenizer, training=False):
        self.records, self.training, self.epoch = records, training, 0
        self.sources = [tokenizer(row['source'], truncation=False) for row in records]
        self.targets = [[tokenizer(text_target=make_joint_target(row['error_type'], ref),
                                  truncation=False)['input_ids'] for ref in row['references']]
                        for row in records]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        refs = self.targets[index]
        target = random.Random(SEED + self.epoch * 1_000_003 + index).choice(refs) if self.training else refs[0]
        return {'input_ids': list(self.sources[index]['input_ids']),
                'attention_mask': list(self.sources[index]['attention_mask']), 'labels': list(target)}

class ReferenceEpochCallback(TrainerCallback):
    def __init__(self, dataset):
        self.dataset = dataset

    def on_epoch_begin(self, args, state, control, **kwargs):
        self.dataset.epoch = int(state.epoch or 0)

def parse_joint_output(raw_text):
    label_match = re.match(r'\A\s*error_?type\s*:\s*(-?\d+)(?=\s|$)', raw_text, re.IGNORECASE)
    correction_match = re.search(r'corrected_?sentence\s*:[ \t]?(.*)\Z', raw_text, re.IGNORECASE | re.DOTALL)
    label = int(label_match.group(1)) if label_match else -1
    if label not in range(5):
        label = -1
    correction = correction_match.group(1) if correction_match else raw_text
    full_match = re.fullmatch(r'\s*error_?type\s*:\s*(-?\d+)\s+corrected_?sentence\s*:[ \t]?(.*)',
                              raw_text, re.IGNORECASE | re.DOTALL)
    return {'raw_prediction': raw_text, 'predicted_error_type': label, 'prediction': correction,
            'format_valid': bool(full_match and label != -1 and correction.strip())}

@torch.inference_mode()
def predict_records(model, records, batch_size, max_length):
    model.eval()
    device = next(model.parameters()).device
    output_rows = []
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        inputs = tokenizer([row['source'] for row in batch], return_tensors='pt', padding=True,
                           truncation=False, return_token_type_ids=False).to(device)
        if inputs['input_ids'].shape[1] > model.config.max_position_embeddings:
            raise ValueError('Input inference vượt giới hạn model; không tự cắt câu.')
        generated = model.generate(**inputs, num_beams=NUM_BEAMS, do_sample=False,
                                   max_length=max_length, early_stopping=True, use_cache=True)
        texts = tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        for row, text, token_ids in zip(batch, texts, generated):
            parsed = parse_joint_output(text)
            ids = token_ids.tolist()
            eos = model.generation_config.eos_token_id
            eos_ids = {eos} if isinstance(eos, int) else set(eos or [])
            # Bỏ decoder-start. EOS ở vị trí cuối có thể do forced_eos khi chạm giới hạn.
            effective = len(ids)
            for offset, token in enumerate(ids[1:], 1):
                if token in eos_ids:
                    effective = offset + 1
                    break
            output_rows.append({**row, **parsed, 'possibly_length_limited': effective >= max_length})
        print(f'Prediction dev: {min(start + batch_size, len(records))}/{len(records)}', flush=True)
    if len(output_rows) != len(records):
        raise RuntimeError('Số prediction không khớp số dòng dev.')
    return output_rows

for text in ['error_type: 1 corrected_sentence: Tôi đang học.',
             'errortype: 1 correctedsentence: Tôi đang học.']:
    assert parse_joint_output(text)['format_valid']
assert parse_joint_output('error_type: 8 corrected_sentence: X')['predicted_error_type'] == -1
assert not parse_joint_output('error_type: 0 corrected_sentence: ')['format_valid']
# Kiểm tra tokenizer thực sự giữ format target đọc được trước khi bắt đầu train.
for label in range(5):
    target = make_joint_target(label, 'Tôi đang học ở trường.')
    decoded = tokenizer.decode(tokenizer(text_target=target, truncation=False)['input_ids'],
                               skip_special_tokens=True, clean_up_tokenization_spaces=False)
    parsed = parse_joint_output(decoded)
    if parsed['predicted_error_type'] != label or not parsed['format_valid']:
        raise ValueError(f'Tokenizer round-trip làm hỏng format target: {decoded!r}')
print('Đã định nghĩa dataset, inference và kiểm tra parser.', flush=True)


GPU: Tesla T4 | GPU count: 2
Torch: 2.11.0+cu128 | Transformers: 4.57.1
Đang tải tokenizer...


config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

dict.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đã định nghĩa dataset, inference và kiểm tra parser.


## 5. Định nghĩa và kiểm tra metrics

Macro-F1 tính cố định trên 5 nhãn 0..4; Micro-F1 tính cả nhãn invalid -1. EM, GLEU+, ROUGE-L dùng đáp án đầu tiên để giữ giao thức của notebook gốc; `EM_any_reference` chấp nhận bất kỳ đáp án gold nào.

In [5]:
import math
from collections import Counter
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score, confusion_matrix

def btc_tokens(text):
    # Token theo khoảng trắng, giữ hoa/thường và dấu câu.
    return text.split()

def btc_ngrams(tokens, n):
    return Counter(
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    )

def btc_lcs(a, b):
    # LCS: longest common subsequence.
    if len(a) < len(b):
        a, b = b, a

    previous = [0] * (len(b) + 1)

    for x in a:
        current = [0]

        for j, y in enumerate(b, start=1):
            if x == y:
                current.append(previous[j - 1] + 1)
            else:
                current.append(
                    max(previous[j], current[-1])
                )

        previous = current

    return previous[-1]

def btc_score(rows):
    assert rows

    numerators = [0] * 4
    denominators = [0] * 4
    hypothesis_length = 0
    reference_length = 0
    exact_matches = 0
    rouge_sum = 0.0

    for row in rows:
        source = row["source"]
        hypothesis = row["prediction"]

        # Chọn cố định, không chọn reference theo prediction.
        reference = row["references"][0]

        # EM: nguyên chuỗi, không strip/normalize.
        exact_matches += hypothesis == reference

        s = btc_tokens(source)
        h = btc_tokens(hypothesis)
        r = btc_tokens(reference)

        hypothesis_length += len(h)
        reference_length += len(r)

        # ROUGE-L F1 = 2*LCS / (len(h) + len(r)).
        lcs_length = btc_lcs(h, r)
        total_length = len(h) + len(r)

        rouge_sum += (
            2 * lcs_length / total_length
            if total_length else 0.0
        )

        # GLEU+, n = 1, 2, 3, 4.
        for n in range(1, 5):
            source_counts = btc_ngrams(s, n)
            hypothesis_counts = btc_ngrams(h, n)
            reference_counts = btc_ngrams(r, n)

            denominators[n - 1] += sum(
                hypothesis_counts.values()
            )

            for gram, count_h in hypothesis_counts.items():
                match_reference = min(
                    count_h, reference_counts[gram]
                )
                match_source = min(
                    count_h, source_counts[gram]
                )

                penalty = max(
                    0, match_source - match_reference
                )

                # Giữ đóng góp âm theo công thức PDF.
                numerators[n - 1] += (
                    match_reference - penalty
                )

    precisions = [
        numerator / denominator if denominator else 0.0
        for numerator, denominator in zip(
            numerators, denominators
        )
    ]

    if hypothesis_length == 0:
        bp = 0.0
    else:
        bp = math.exp(
            min(
                0.0,
                1 - reference_length / hypothesis_length,
            )
        )

    if all(p > 0 for p in precisions):
        gleu = bp * math.exp(
            sum(math.log(p) for p in precisions) / 4
        )
    else:
        gleu = 0.0

    return {
        "n": len(rows),
        "EM": exact_matches / len(rows),
        "GLEU+": gleu,
        "ROUGE-L": rouge_sum / len(rows),
        "GLEU_BP": bp,
        "GLEU_precisions": precisions,
        "GLEU_numerators": numerators,
        "GLEU_denominators": denominators,
    }



def evaluate_predictions(rows):
    if not rows:
        raise ValueError('Dev prediction rỗng.')
    labels = list(range(5))
    y_true = [row['error_type'] for row in rows]
    y_pred = [row['predicted_error_type'] for row in rows]
    if not all(type(x) is int and x in labels for x in y_true):
        raise ValueError('Gold labels không thuộc 0..4.')
    if not all(type(x) is int and x in labels + [-1] for x in y_pred):
        raise ValueError('Prediction labels không hợp lệ.')
    p, r, f, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    classification = {
        'Macro-F1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'Micro-F1': float(f1_score(y_true, y_pred, labels=labels + [-1], average='micro', zero_division=0)),
        'Accuracy': float(accuracy_score(y_true, y_pred)),
        'invalid_label_count': y_pred.count(-1),
        'invalid_format_count': sum(not row['format_valid'] for row in rows),
        'per_class': [{'error_type': label, 'Precision': float(p[i]), 'Recall': float(r[i]),
                       'F1': float(f[i]), 'support': int(support[i])} for i, label in enumerate(labels)],
        'confusion_labels': labels + [-1],
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=labels + [-1]).tolist(),
    }
    scopes = {'dev_all': rows, 'dev_errors_only': [r for r in rows if r['error_type'] != 0]}
    scopes.update({f'dev_gold_type_{label}': [r for r in rows if r['error_type'] == label] for label in labels})
    return {'n': len(rows), 'classification': classification,
            'correction': {scope: btc_score(part) if part else {} for scope, part in scopes.items()},
            'EM_any_reference': sum(r['prediction'] in r['references'] for r in rows) / len(rows),
            'unchanged_rate': sum(r['prediction'] == r['source'] for r in rows) / len(rows)}

# Perfect correction, >=4 tokens để GLEU+ có đủ 4 bậc n-gram.
test_score = btc_score([{'source': 'Tôi đi học vào hôm qua .',
                        'prediction': 'Tôi đã đi học vào hôm qua .',
                        'references': ['Tôi đã đi học vào hôm qua .']}])
for metric in ['EM', 'GLEU+', 'ROUGE-L']:
    assert abs(test_score[metric] - 1.0) < 1e-12
print('Đã kiểm tra metric correction. Macro/Micro-F1 sẽ tính trên prediction dev sau training.', flush=True)


Đã kiểm tra metric correction. Macro/Micro-F1 sẽ tính trên prediction dev sau training.


## 6. Fine-tune một model và sinh prediction dev

Checkpoint lưu đủ optimizer/scheduler/RNG để resume trong cùng cấu hình. Run có dấu vân tay dữ liệu, code và hyperparameters; đổi dữ liệu/cấu hình tạo run riêng. Không xóa checkpoint khi train xong.

In [6]:
def complete_model_exists(directory):
    directory = Path(directory)
    weights = list(directory.glob('*.safetensors')) + list(directory.glob('pytorch_model*.bin'))
    return ((directory / 'config.json').is_file() and bool(weights)
            and all(path.stat().st_size > 0 for path in weights)
            and (directory / 'tokenizer_config.json').is_file())

def find_resume_checkpoint(run_dir):
    from safetensors import safe_open
    candidates = [path for path in (run_dir / 'checkpoints').glob('checkpoint-*')
                  if path.is_dir() and path.name.split('-')[-1].isdigit()]
    for checkpoint in sorted(candidates, key=lambda p: int(p.name.split('-')[-1]), reverse=True):
        required = ['trainer_state.json', 'optimizer.pt', 'scheduler.pt', 'rng_state.pth',
                    'model.safetensors', 'config.json']
        if not all((checkpoint / name).is_file() and (checkpoint / name).stat().st_size > 0 for name in required):
            print('Bỏ qua checkpoint chưa đầy đủ:', checkpoint.name, flush=True)
            continue
        try:
            read_json(checkpoint / 'trainer_state.json')
            with safe_open(str(checkpoint / 'model.safetensors'), framework='pt', device='cpu') as saved:
                if not list(saved.keys()):
                    continue
        except Exception as exc:
            print('Không đọc được checkpoint:', checkpoint.name, type(exc).__name__, flush=True)
            continue
        return str(checkpoint)
    return None

def train_btc_dev():
    set_seed(SEED)
    gc.collect()
    torch.cuda.empty_cache()
    print('Đang tokenize train/dev, giữ toàn bộ references...', flush=True)
    train_dataset = CorrectionDataset(train_records, tokenizer, training=True)
    dev_dataset = CorrectionDataset(dev_records, tokenizer, training=False)
    print('Đang tải BARTpho pretrained...', flush=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model_limit = model.config.max_position_embeddings
    length_issues = []
    lengths = []
    for name, dataset in [('train', train_dataset), ('dev', dev_dataset)]:
        for row, source, targets in zip(dataset.records, dataset.sources, dataset.targets):
            for field, length in [('Input', len(source['input_ids']))] + [
                    (f'joint_reference_{i}', len(ids)) for i, ids in enumerate(targets)]:
                lengths.append(length)
                if math.ceil(length / 8) * 8 > model_limit:
                    length_issues.append({'dataset': name, 'ID': row['id'], 'field': field, 'tokens': length})
    write_json(DATA_DIR / 'token_length_issues.json', length_issues)
    if length_issues:
        raise ValueError(f'Có câu vượt {model_limit} tokens. Xem token_length_issues.json; code không tự cắt gold.')
    longest = max(lengths)
    generation_max_length = min(model_limit, max(256, math.ceil((longest + 32) / 8) * 8))
    run_config = {'pipeline_version': PIPELINE_VERSION, 'model': MODEL_NAME,
                  'data_sha256': prepared['manifest']['sha256'], 'seed': SEED, 'epochs_max': EPOCHS,
                  'learning_rate': LEARNING_RATE, 'train_batch_size': TRAIN_BATCH_SIZE,
                  'gradient_accumulation_steps': GRAD_ACCUMULATION, 'eval_batch_size': EVAL_BATCH_SIZE,
                  'patience': EARLY_STOPPING_PATIENCE, 'num_beams': NUM_BEAMS,
                  'fp16': FP16, 'bf16': BF16, 'torch_version': str(torch.__version__),
                  'transformers_version': transformers.__version__, 'gpu_count': torch.cuda.device_count(),
                  'generation_max_length': generation_max_length,
                  'best_checkpoint_metric': 'dev first-reference joint-target loss',
                  'reference_training': 'one seeded random reference per record per epoch',
                  'code_sha256': NOTEBOOK_CODE_SHA256}
    run_id = sha256_bytes(json.dumps(run_config, sort_keys=True).encode())
    run_dir = OUTPUT_ROOT / 'runs' / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    config_path = run_dir / 'run_config.json'
    if config_path.is_file() and read_json(config_path) != run_config:
        raise RuntimeError('Cấu hình run không khớp. Không dùng kết quả cũ.')
    write_json(config_path, run_config)
    marker = run_dir / 'completed.json'
    if marker.is_file() and complete_model_exists(run_dir / 'best_model'):
        saved = read_json(marker)
        valid_saved = saved.get('run_id') == run_id
        for filename, digest in saved.get('output_sha256', {}).items():
            path = run_dir / filename
            valid_saved = valid_saved and path.is_file() and sha256_bytes(path.read_bytes()) == digest
        valid_saved = valid_saved and {'dev_predictions.json', 'dev_metrics.json', 'training_log.json'} <= set(saved.get('output_sha256', {}))
        if valid_saved:
            predictions = read_json(run_dir / 'dev_predictions.json')
            if len(predictions) == len(dev_records) and all(
                    {k: row[k] for k in ('id', 'dataset', 'source', 'error_type', 'references')} == gold
                    for row, gold in zip(predictions, dev_records)):
                del model
                gc.collect()
                torch.cuda.empty_cache()
                print('Run này đã hoàn thành; dùng model/prediction đã lưu:', run_dir, flush=True)
                return run_dir, predictions, read_json(run_dir / 'dev_metrics.json')
        raise RuntimeError('Output đã hoàn thành bị thay đổi hoặc không khớp dev. Giữ lại run cũ và đổi OUTPUT_ROOT để chạy mới.')
    model.config.use_cache = False
    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True,
                                     pad_to_multiple_of=8, label_pad_token_id=-100)
    # Kiểm tra padding/labels trước vòng train dài.
    collator([train_dataset[0], dev_dataset[0]])
    n_gpu = max(1, torch.cuda.device_count())
    steps_per_epoch = math.ceil(math.ceil(len(train_records) / (TRAIN_BATCH_SIZE * n_gpu)) / GRAD_ACCUMULATION)
    warmup_steps = max(1, round(0.1 * EPOCHS * steps_per_epoch))
    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir / 'checkpoints'), num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE, gradient_accumulation_steps=GRAD_ACCUMULATION,
        weight_decay=0.01, warmup_steps=warmup_steps, lr_scheduler_type='linear', max_grad_norm=1.0,
        eval_strategy='epoch', save_strategy='epoch', logging_strategy='steps', logging_steps=5,
        logging_first_step=True, disable_tqdm=False, load_best_model_at_end=True,
        metric_for_best_model='eval_loss', greater_is_better=False,
        save_total_limit=2, save_only_model=False, save_safetensors=True,
        restore_callback_states_from_checkpoint=True,
        fp16=FP16, bf16=BF16, gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False}, predict_with_generate=False,
        prediction_loss_only=True, dataloader_num_workers=0,
        report_to='none', optim='adamw_torch', seed=SEED, data_seed=SEED)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_dataset, eval_dataset=dev_dataset,
                            processing_class=tokenizer, data_collator=collator,
                            callbacks=[ReferenceEpochCallback(train_dataset),
                                       EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)])
    checkpoint = find_resume_checkpoint(run_dir) if RESUME_IF_AVAILABLE else None
    if not RESUME_IF_AVAILABLE and any((run_dir / 'checkpoints').glob('checkpoint-*')):
        raise ValueError('Run đã có checkpoint. Bật RESUME_IF_AVAILABLE hoặc đổi OUTPUT_ROOT trước khi train mới.')
    print('Train:', len(train_records), '| Dev:', len(dev_records), '| Token dài nhất:', longest, flush=True)
    print('Resume:', checkpoint, '| Run:', run_dir, flush=True)
    trainer.train(resume_from_checkpoint=checkpoint)
    dev_loss = float(trainer.evaluate()['eval_loss'])
    model = trainer.model
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    model.generation_config.max_length = generation_max_length
    model.generation_config.num_beams = NUM_BEAMS
    model.generation_config.do_sample = False
    model.generation_config.early_stopping = True
    trainer.save_model(str(run_dir / 'best_model'))
    tokenizer.save_pretrained(str(run_dir / 'best_model'))
    trainer.save_state()
    write_json(run_dir / 'training_log.json', trainer.state.log_history)
    print('Đã lưu best_model. Đang sinh prediction trên toàn bộ dev...', flush=True)
    prediction_rows = predict_records(model, dev_records, EVAL_BATCH_SIZE, generation_max_length)
    write_json(run_dir / 'dev_predictions.json', prediction_rows)
    metrics = evaluate_predictions(prediction_rows)
    metrics['eval_loss'] = dev_loss
    metrics['best_checkpoint'] = trainer.state.best_model_checkpoint
    metrics['best_metric'] = float(trainer.state.best_metric)
    metrics['possibly_length_limited_count'] = sum(r['possibly_length_limited'] for r in prediction_rows)
    write_json(run_dir / 'dev_metrics.json', metrics)
    write_json(run_dir / 'completed.json', {'run_id': run_id, 'output_sha256': {
        name: sha256_bytes((run_dir / name).read_bytes()) for name in
        ['dev_predictions.json', 'dev_metrics.json', 'training_log.json']}})
    print('Hoàn thành. Model và kết quả:', run_dir, flush=True)
    del trainer, model, collator
    gc.collect()
    torch.cuda.empty_cache()
    return run_dir, prediction_rows, metrics

RUN_DIR, dev_prediction_rows, dev_metrics = train_btc_dev()


Đang tokenize train/dev, giữ toàn bộ references...
Đang tải BARTpho pretrained...


pytorch_model.bin:   0%|          | 0.00/526M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

Train: 1857 | Dev: 90 | Token dài nhất: 71
Resume: None | Run: /kaggle/working/bartpho_btc_dev/runs/a3074f544e0e236e93ea5f4812be3a511ef20834cd7af03bdc37efac0d8ca173


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,0.338800,0.403728
2,0.238300,0.349970
3,0.130200,0.353192
4,0.102600,0.332317
5,0.156900,0.337101
6,0.062800,0.352821
7,0.041900,0.355653


/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Đã lưu best_model. Đang sinh prediction trên toàn bộ dev...
Prediction dev: 2/90
Prediction dev: 4/90
Prediction dev: 6/90
Prediction dev: 8/90
Prediction dev: 10/90
Prediction dev: 12/90
Prediction dev: 14/90
Prediction dev: 16/90
Prediction dev: 18/90
Prediction dev: 20/90
Prediction dev: 22/90
Prediction dev: 24/90
Prediction dev: 26/90
Prediction dev: 28/90
Prediction dev: 30/90
Prediction dev: 32/90
Prediction dev: 34/90
Prediction dev: 36/90
Prediction dev: 38/90
Prediction dev: 40/90
Prediction dev: 42/90
Prediction dev: 44/90
Prediction dev: 46/90
Prediction dev: 48/90
Prediction dev: 50/90
Prediction dev: 52/90
Prediction dev: 54/90
Prediction dev: 56/90
Prediction dev: 58/90
Prediction dev: 60/90
Prediction dev: 62/90
Prediction dev: 64/90
Prediction dev: 66/90
Prediction dev: 68/90
Prediction dev: 70/90
Prediction dev: 72/90
Prediction dev: 74/90
Prediction dev: 76/90
Prediction dev: 78/90
Prediction dev: 80/90
Prediction dev: 82/90
Prediction dev: 84/90
Prediction dev: 86/9

## 7. Xem Macro/Micro-F1 và tải kết quả

ZIP mặc định chứa best_model, tokenizer, prediction, metrics và dữ liệu đã lọc. Đủ để inference lần sau. Muốn đóng gói checkpoint để tiếp tục training, bật `INCLUDE_CHECKPOINTS_IN_ZIP=True` rồi chạy lại cell này. **Save Version → Save & Run All** trên Kaggle để giữ Output.

In [7]:
import csv
import zipfile
from IPython.display import FileLink, display

def write_csv(path, rows, fieldnames):
    path = Path(path)
    with path.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

def export_run(run_dir, data_dir, rows, metrics):
    write_json(run_dir / 'classification_per_class.json', metrics['classification']['per_class'])
    write_csv(run_dir / 'classification_per_class.csv', metrics['classification']['per_class'],
              ['error_type', 'Precision', 'Recall', 'F1', 'support'])
    cm_labels = metrics['classification']['confusion_labels']
    cm = metrics['classification']['confusion_matrix']
    write_csv(run_dir / 'confusion_matrix.csv', [{'gold': label, **dict(zip(map(str, cm_labels), counts))}
              for label, counts in zip(cm_labels, cm)], ['gold'] + list(map(str, cm_labels)))
    correction_rows = [{'scope': scope, **{k: value[k] for k in ['n', 'EM', 'GLEU+', 'ROUGE-L']}}
                       for scope, value in metrics['correction'].items() if value]
    write_csv(run_dir / 'correction_report_0_1.csv', correction_rows, ['scope', 'n', 'EM', 'GLEU+', 'ROUGE-L'])
    flat_rows = [{k: row[k] for k in ['id', 'source', 'error_type', 'predicted_error_type',
                                     'prediction', 'raw_prediction', 'format_valid', 'possibly_length_limited']}
                 for row in rows]
    write_csv(run_dir / 'dev_predictions.csv', flat_rows, list(flat_rows[0]))
    protocol = {'official_scorer': False, 'evaluation_data': 'Entire provided dev, original row order retained',
                'best_model_selection': 'lowest dev loss, first reference joint target',
                'class_labels_macro': [0, 1, 2, 3, 4], 'micro_labels_including_invalid': [0, 1, 2, 3, 4, -1],
                'macro_absent_class': 'included as F1=0', 'invalid_label': -1,
                'tokenization': 'whitespace split; case sensitive',
                'EM': 'raw string equality to references[0]',
                'EM_any_reference': 'raw string equality to any gold reference',
                'label0_model_reference': 'Input as identity target; original gold [] is retained in dev_preserved.json',
                'reference_selection_GLEU_ROUGE': 'references[0], not selected using prediction',
                'GLEU+': 'custom corpus implementation retained from original notebook; n=1..4; no smoothing',
                'ROUGE-L': 'mean sentence LCS F1 on first reference', 'saved_score_scale': '0..1',
                'Input_overlap_match': prepared['manifest']['match_rule'],
                'near_duplicate_filter': False}
    write_json(run_dir / 'evaluation_protocol.json', protocol)
    readme = ('BARTpho BTC train / new dev\n\n'
              'best_model/: model + tokenizer; load with AutoTokenizer/AutoModelForSeq2SeqLM.from_pretrained.\n'
              'dev_predictions.json/csv: one prediction per dev record; label -1 means invalid generated label.\n'
              'dev_metrics.json: correction and classification metrics (0..1).\n'
              'data/train_filtered.json: original BTC records, excluding matching dev Inputs.\n'
              'data/dev_preserved.json: all original dev rows and references preserved.\n'
              'data/removed_train_dev_overlap.json and overlap_log.json: removed records and matched dev IDs.\n'
              'data/data_issues.json: automatic warnings requiring manual review.\n'
              'Evaluation is not an official BTC scorer; see evaluation_protocol.json.\n'
              'ZIP excludes checkpoints by default; best_model is sufficient for inference.\n'
              'Training resume requires retained checkpoints with optimizer/scheduler/RNG state.\n')
    (run_dir / 'README.txt').write_text(readme, encoding='utf-8')
    archive_path = OUTPUT_ROOT / f'bartpho_btc_dev_{run_dir.name[:16]}.zip'
    temporary = archive_path.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(run_dir.rglob('*')):
            if not path.is_file():
                continue
            relative = path.relative_to(run_dir)
            if not INCLUDE_CHECKPOINTS_IN_ZIP and relative.parts[0] == 'checkpoints':
                continue
            archive.write(path, str(relative))
        for path in sorted(data_dir.glob('*.json')):
            archive.write(path, 'data/' + path.name)
    temporary.replace(archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        names = set(archive.namelist())
        assert 'best_model/config.json' in names and 'dev_predictions.json' in names
        assert 'data/train_filtered.json' in names and 'data/dev_preserved.json' in names
    return archive_path

print('===== METRIC DEV — THANG 0..1 =====')
classification = dev_metrics['classification']
for metric in ['Macro-F1', 'Micro-F1', 'Accuracy', 'invalid_label_count', 'invalid_format_count']:
    print(f'{metric}: {classification[metric]}')
for metric in ['EM', 'GLEU+', 'ROUGE-L']:
    print(f"{metric} (first reference): {dev_metrics['correction']['dev_all'][metric]:.6f}")
print(f"EM_any_reference: {dev_metrics['EM_any_reference']:.6f}")
print('possibly_length_limited_count:', dev_metrics.get('possibly_length_limited_count', 0))
print('Best dev loss:', dev_metrics['eval_loss'])
print('Đang đóng gói model, dữ liệu đã lọc và prediction...', flush=True)
ARCHIVE_PATH = export_run(RUN_DIR, DATA_DIR, dev_prediction_rows, dev_metrics)
print('ZIP:', ARCHIVE_PATH, '| GB:', round(ARCHIVE_PATH.stat().st_size / 2**30, 3))
for path in [RUN_DIR / 'dev_predictions.json', RUN_DIR / 'dev_metrics.json',
             DATA_DIR / 'train_filtered.json', DATA_DIR / 'removed_train_dev_overlap.json', ARCHIVE_PATH]:
    display(FileLink(str(path)))
print('Kaggle: Save Version → Save & Run All để giữ Output sau khi session kết thúc.')


===== METRIC DEV — THANG 0..1 =====
Macro-F1: 0.34615384615384615
Micro-F1: 0.6111111111111112
Accuracy: 0.6111111111111112
invalid_label_count: 0
invalid_format_count: 0
EM (first reference): 0.433333
GLEU+ (first reference): 0.638001
ROUGE-L (first reference): 0.879005
EM_any_reference: 0.433333
possibly_length_limited_count: 0
Best dev loss: 0.3323172628879547
Đang đóng gói model, dữ liệu đã lọc và prediction...
ZIP: /kaggle/working/bartpho_btc_dev/bartpho_btc_dev_a3074f544e0e236e.zip | GB: 0.497


/kaggle/working/bartpho_btc_dev/runs/a3074f544e0e236e93ea5f4812be3a511ef20834cd7af03bdc37efac0d8ca173/dev_predictions.json

/kaggle/working/bartpho_btc_dev/runs/a3074f544e0e236e93ea5f4812be3a511ef20834cd7af03bdc37efac0d8ca173/dev_metrics.json

/kaggle/working/bartpho_btc_dev/data/5bdb5eb56670b6dab4f78ed9b156047d1b3da6ef90e1c89a01594d414b456b03/train_filtered.json

/kaggle/working/bartpho_btc_dev/data/5bdb5eb56670b6dab4f78ed9b156047d1b3da6ef90e1c89a01594d414b456b03/removed_train_dev_overlap.json

/kaggle/working/bartpho_btc_dev/bartpho_btc_dev_a3074f544e0e236e.zip

Kaggle: Save Version → Save & Run All để giữ Output sau khi session kết thúc.


## 8. Ví dụ dùng lại model (tùy chọn)

In [8]:
# CELL TÙY CHỌN — chạy sau training để minh họa dùng lại model đã lưu.
# Sau khi tải ZIP lên Kaggle ở lần sau, gán MODEL_DIR vào thư mục best_model đó.
# Không cần chạy lại các cell training để inference; cần cùng phiên bản thư viện đã pin.
RUN_INFERENCE_EXAMPLE = False
if RUN_INFERENCE_EXAMPLE:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    MODEL_DIR = RUN_DIR / 'best_model'
    inference_tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), use_fast=False)
    inference_model = AutoModelForSeq2SeqLM.from_pretrained(str(MODEL_DIR))
    inference_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    inference_model.to(inference_device).eval()
    texts = ['Tôi đi học vào hôm qua.']
    inputs = inference_tokenizer(texts, return_tensors='pt', padding=True, truncation=False,
                                 return_token_type_ids=False).to(inference_device)
    if inputs['input_ids'].shape[1] > inference_model.config.max_position_embeddings:
        raise ValueError('Câu quá dài; cần xử lý riêng, không tự cắt.')
    with torch.inference_mode():
        outputs = inference_model.generate(**inputs, num_beams=NUM_BEAMS, do_sample=False,
                                            max_length=inference_model.generation_config.max_length,
                                            early_stopping=True)
    for text in inference_tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                                 clean_up_tokenization_spaces=False):
        print(parse_joint_output(text))
    del inference_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [9]:
import pandas as pd
from IPython.display import display

c = dev_metrics["classification"]
labels = c["confusion_labels"]
cm = c["confusion_matrix"]
n = sum(sum(row) for row in cm)

report = []
for row in c["per_class"]:
    label = row["error_type"]
    i = labels.index(label)

    tp = cm[i][i]
    fn = sum(cm[i]) - tp
    fp = sum(cm[j][i] for j in range(len(labels))) - tp
    tn = n - tp - fn - fp

    report.append({
        "Error_Type_Id": label,
        "Accuracy_OvR": (tp + tn) / n,
        "Precision": row["Precision"],
        "Recall": row["Recall"],
        "F1": row["F1"],
        "Support": row["support"],
    })

df = pd.DataFrame(report)
display(df.round(4))
df.to_csv(
    RUN_DIR / "error_type_APRF1.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Overall Accuracy:", c["Accuracy"])
print("Macro-F1:", c["Macro-F1"])
print("Micro-F1:", c["Micro-F1"])

,Error_Type_Id,Accuracy_OvR,Precision,Recall,F1,Support
0,0,0.6889,0.6129,0.9048,0.7308,42
1,1,0.7778,0.7143,0.5172,0.6000,29
2,2,0.9111,0.0000,0.0000,0.0000,6
3,3,0.9333,0.5000,0.3333,0.4000,6
4,4,0.9111,0.0000,0.0000,0.0000,7


Overall Accuracy: 0.6111111111111112
Macro-F1: 0.34615384615384615
Micro-F1: 0.6111111111111112


In [10]:
ids = sorted(
    row["id"]
    for row in dev_prediction_rows
    if row["error_type"] != 0
    and row["prediction"] == row["source"]
)

print("Số câu lỗi bị giữ nguyên:", len(ids))
print("ID:", ids)

Số câu lỗi bị giữ nguyên: 29
ID: [15, 23, 26, 27, 41, 49, 51, 55, 73, 74, 81, 82, 83, 84, 96, 100, 115, 125, 135, 138, 150, 155, 158, 159, 174, 184, 444, 447, 456]
